# PEEL Phase 2 -- Informative Sentence Selection & Condensation

Selects the most informative sentences per Phase 1 cluster (lexical
density + stem/n-gram matching), then optionally generates an LLM
condensation of the corpus at a chosen rate and verifies it (injection
taxonomy classification, verbatim-overlap scan, cluster coverage). See the
root [README.md](../README.md) for the full pipeline overview.

Set `CORPUS_NAME` below to a corpus that has already been run through
Phase 1 -- this notebook reads `data/<CORPUS_NAME>/phase1/<CORPUS_NAME>-phase1_state.json`
directly, no manual copying required.

The condensation cells below require a locally running
[Ollama](https://ollama.com) instance with at least one model pulled.

In [ ]:
# CONFIG

import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import spacy
from nltk.stem import PorterStemmer

from common.paths import CorpusPaths
from common.decisions import DecisionLog
from common import ollama_client, voyant_notebook, standalone_report
from phase2 import pipeline, condense, condensation_report

CORPUS_NAME = "Boisseau"  # only line to edit to switch corpus
TOP_N = 10                 # most insightful sentences kept per cluster
LEXICAL_DENSITY_PERCENTILE = 75

paths = CorpusPaths(CORPUS_NAME)
paths.ensure_dirs()
decisions = DecisionLog(CORPUS_NAME, phase="phase2")

In [ ]:
# LOAD SPACY, CORPUS TEXT, AND PHASE 1 STATE

print("Loading spaCy...")
nlp = spacy.load("en_core_web_sm")
stemmer = PorterStemmer()
print("Done.")

phase1_state = pipeline.load_phase1_state(paths.phase1_state_json())

with open(paths.raw_txt(), encoding="utf8") as f:
    text = f.read()

In [ ]:
# SELECT REPRESENTATIVE SENTENCES PER CLUSTER

enriched_state = pipeline.enrich_with_informative_sentences(
    phase1_state, text, nlp, stemmer,
    top_n=TOP_N, density_percentile=LEXICAL_DENSITY_PERCENTILE,
)

In [ ]:
# SAVE JSON

pipeline.save_informative_sentences(enriched_state, paths.phase2_output_json())
print(f"\nSaved enriched JSON to\n{paths.phase2_output_json()}")

In [ ]:
# CONDENSATION CONFIG
# Every choice here is appended to data/<CORPUS_NAME>/decisions/phase2_decisions.jsonl
# via `decisions.record(...)` for later audit -- it does not change the interactive flow.

RATES, OLLAMA_MODEL, MAX_TRIALS = condense.run_condensation_setup(decisions)

In [ ]:
# GENERATE CONDENSATION
# For each requested rate: attempts generation up to MAX_TRIALS times; if
# none land within +/-5% of the target word count, asks whether to retry
# with the full source text included in the prompt. Every trial and the
# escalation choice are logged via `decisions.record(...)`.

ordered_sentences = condense.gather_ordered_informative_sentences(enriched_state)
cluster_key_terms = condense.gather_cluster_key_terms(enriched_state)

condensed_texts = {}

for rate in RATES:

    target_words = condense.target_word_count(text, rate)
    result = condense.run_generate_for_rate(
        rate, ordered_sentences, cluster_key_terms, target_words, CORPUS_NAME,
        text, OLLAMA_MODEL, MAX_TRIALS, decisions,
    )

    if result["text"] is None:
        continue

    condensed_texts[rate] = result["text"]

    condensed_path = paths.condensation_paths(rate)["condensed_text"]
    with open(condensed_path, "w", encoding="utf-8") as f:
        f.write(result["text"])
    print(f"\nSaved condensed text to {condensed_path}")

In [ ]:
# INJECTION ANALYSIS
# Classifies every non-verbatim sentence in each condensation (F/T/R/C),
# flags borderline calls, and lets you reclassify them -- each choice
# logged via `decisions.record(...)`. Classification is heuristic, not
# LLM-judged; see README.md for the taxonomy and this limitation.

injection_results = {}

for rate, condensed_text in condensed_texts.items():

    print(f"\n=== Injection analysis: {rate}% condensation ===")

    all_spans, source_sentences = condense.classify_condensation(condensed_text, text, nlp)
    stats = condense.compute_injection_stats(all_spans, condensed_text, text)
    borderline_flags = condense.flag_borderline_classifications(all_spans)

    print(f"Spans classified: {len(all_spans)}")
    print(f"Non-injected word share: {stats['non_injected_pct']}%")
    print(f"Verbatim-overlap scan: {stats['verbatim_overlap_pct']}%")

    condense.run_injection_review(all_spans, borderline_flags, rate, decisions)

    injection_results[rate] = {
        "all_spans": all_spans,
        "source_sentences": source_sentences,
        "stats": stats,
        "borderline_flags": borderline_flags,
    }

In [ ]:
# CLUSTER COVERAGE
# Per Phase 1 cluster: this cluster's share of all cluster-vocabulary
# token occurrences in the source (the target) vs. the same share in the
# condensation (actual) -- see README.md for how OK/WARN/DARK are decided.

coverage_results = {}

for rate, condensed_text in condensed_texts.items():

    coverage_results[rate] = condense.compute_cluster_coverage(phase1_state, condensed_text, text, stemmer)

    print(f"\n=== Cluster coverage: {rate}% condensation ===")
    for name, r in coverage_results[rate].items():
        print(f"  {name}: target {r['target']}%, actual {r['actual']}% ({r['delta']:+.1f}pp) -- {r['status']}")

In [ ]:
# BUILD & SAVE REPORTS
# Writes, per rate: the condensed text (already saved above), a Spyral-
# paste-ready HTML fragment, a standalone browser preview, a plain-text
# verification report, a markup-free plain summary, and the injection
# report JSON.

fragments_by_rate = {}

for rate, condensed_text in condensed_texts.items():

    result = injection_results[rate]
    all_spans = result["all_spans"]
    source_sentences = result["source_sentences"]
    stats = result["stats"]
    borderline_flags = result["borderline_flags"]
    coverage = coverage_results[rate]

    fragment = condensation_report.build_condensation_fragment(
        condensed_text, all_spans, source_sentences, coverage,
        corpus_name=CORPUS_NAME, rate=rate,
        source_word_count=condense.count_words(text),
        phase1_json_name=paths.phase1_state_json().name,
    )
    fragments_by_rate[rate] = fragment

    preview = condensation_report.build_standalone_preview(fragment, CORPUS_NAME)
    sanity_issues = condense.check_condensation_sanity(condensed_text, ordered_sentences)
    report = condensation_report.build_human_report(
        all_spans, borderline_flags, coverage,
        stats["verbatim_overlap_pct"], stats["non_injected_pct"], sanity_issues,
    )
    blocks = condensation_report.parse_condensed_blocks(condensed_text)
    summary = condensation_report.build_plain_summary(blocks)

    injection_report_data = {
        "corpus": CORPUS_NAME, "rate": rate,
        "spans": all_spans, "borderline_flags": borderline_flags,
        "coverage": coverage, "stats": stats,
    }

    output_paths = paths.condensation_paths(rate)
    condensation_report.save_condensation_outputs(
        output_paths, fragment, preview, report, summary, injection_report_data,
    )

    print(f"\n=== {rate}% condensation outputs ===")
    for label, p in output_paths.items():
        if label != "condensed_text":
            print(f"  {label}: {p}")

In [ ]:
# BUILD VOYANT NOTEBOOK & STANDALONE REPORT
# Fills resources/PEEL-TemplateSN.html's known placeholders with this
# corpus's Phase 1 + Phase 2 results -- a ready-to-use Voyant Spyral
# notebook per rate (the shared template itself is never modified) -- and
# also builds a parallel, non-Voyant standalone HTML report showing the
# same information (condensation + Phase 1 clusters + summary) with its
# own clean styling, for anyone who doesn't use Voyant.

for rate, fragment in fragments_by_rate.items():

    voyant_html = voyant_notebook.build_voyant_notebook(
        enriched_state, {rate: fragment}, CORPUS_NAME, paths,
    )
    voyant_path = paths.voyant_notebook_path(rate)
    voyant_notebook.save_voyant_notebook(voyant_html, voyant_path)
    print(f"Voyant notebook written to {voyant_path}")

    report_html = standalone_report.build_standalone_report(
        enriched_state, fragment, CORPUS_NAME, rate, paths,
    )
    report_path = paths.standalone_report_path(rate)
    standalone_report.save_standalone_report(report_html, report_path)
    print(f"Standalone report written to {report_path}")